<a href="https://colab.research.google.com/github/EduardoJBenavidesC/Proyecto-Completo/blob/main/TelecomX_LATAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#📌 Extracción

### Preparación del ambiente.

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Carga del dataSet.

In [28]:
df_telecom = pd.read_json('TelecomX_Data.json')
df_telecom.head(5)



,customerID,Churn,customer,phone,internet,account
0,0002-ORFBO,No,"{'gender': 'Female', 'SeniorCitizen': 0, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'One year', 'PaperlessBilling': '..."
1,0003-MKNFE,No,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'Yes'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
2,0004-TLHLJ,Yes,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
3,0011-IGKFF,Yes,"{'gender': 'Male', 'SeniorCitizen': 1, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
4,0013-EXCHZ,Yes,"{'gender': 'Female', 'SeniorCitizen': 1, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."


#### Nota 1: El archivo .json tiene multiples columnas anidadas ==> aplicar .normalize para poder aplanar el dataset.

In [30]:
print(df_telecom.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7267 entries, 0 to 7266
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   customerID  7267 non-null   object
 1   Churn       7267 non-null   object
 2   customer    7267 non-null   object
 3   phone       7267 non-null   object
 4   internet    7267 non-null   object
 5   account     7267 non-null   object
dtypes: object(6)
memory usage: 340.8+ KB
None


#🔧 Transformación

In [31]:
# Convertir a dicionario para normalizar las columnas anidadas de una sola vez.
dict_data = df_telecom.to_dict(orient='records')
df_telecom_norm = pd.json_normalize(dict_data, sep='.')
df_telecom_norm.head()

,customerID,Churn,customer.gender,customer.SeniorCitizen,customer.Partner,customer.Dependents,customer.tenure,phone.PhoneService,phone.MultipleLines,internet.InternetService,...,internet.OnlineBackup,internet.DeviceProtection,internet.TechSupport,internet.StreamingTV,internet.StreamingMovies,account.Contract,account.PaperlessBilling,account.PaymentMethod,account.Charges.Monthly,account.Charges.Total
0,0002-ORFBO,No,Female,0,Yes,Yes,9,Yes,No,DSL,...,Yes,No,Yes,Yes,No,One year,Yes,Mailed check,65.6,593.3
1,0003-MKNFE,No,Male,0,No,No,9,Yes,Yes,DSL,...,No,No,No,No,Yes,Month-to-month,No,Mailed check,59.9,542.4
2,0004-TLHLJ,Yes,Male,0,No,No,4,Yes,No,Fiber optic,...,No,Yes,No,No,No,Month-to-month,Yes,Electronic check,73.9,280.85
3,0011-IGKFF,Yes,Male,1,Yes,No,13,Yes,No,Fiber optic,...,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,98.0,1237.85
4,0013-EXCHZ,Yes,Female,1,Yes,No,3,Yes,No,Fiber optic,...,No,No,Yes,Yes,No,Month-to-month,Yes,Mailed check,83.9,267.4


### Revisión de la calidad de los datos en el contexto del negocio y objetivos.

In [32]:
# Identifición de Nulos Reales (NaN/None)
nulos_reales = df_telecom_norm.isnull().sum()

# Identifición de Celdas que solo contienen espacios.
espacios_blanco = df_telecom_norm.apply(lambda x: x.str.strip().eq('').sum() if x.dtype == 'object' else 0)

# Identificación valores únicos, para ver categorías ruidosas o IDs duplicados)
valores_unicos = df_telecom_norm.nunique()

# Reporte
auditoria_df_telecom_norm = pd.DataFrame({
    'Tipo_Dato': df_telecom_norm.dtypes,
    'Nulos_NaN': nulos_reales,
    'Espacios_Blanco': espacios_blanco,
    'Valores_Unicos': valores_unicos
})

# Mostrar solo columnas que presentan algún problema de los vistos.
alertas = auditoria_df_telecom_norm[(auditoria_df_telecom_norm['Nulos_NaN'] > 0) | (auditoria_df_telecom_norm['Espacios_Blanco'] > 0)]

print('*** Reporte de Auditoría de Calidad ***')
if alertas.empty:
    print('No se detectaron nulos ni espacios en blanco.')
else:
    display(alertas.style.set_caption('Columnas con Incoherencias Detectadas'))

# Verificación de duplicados en la llave primaria (customerID)
duplicados_id = df_telecom_norm['customerID'].duplicated().sum()
print(f'\nRegistros duplicados por ID de Cliente: {duplicados_id}')

*** Reporte de Auditoría de Calidad ***


,Tipo_Dato,Nulos_NaN,Espacios_Blanco,Valores_Unicos
Churn,object,0,224,3
account.Charges.Total,object,0,11,6531



Registros duplicados por ID de Cliente: 0


####  Nota 2: La auditoría técnica identificó 11 registros con espacios en blanco en account.Charges.Total y 2 en Churn.

La inconsistencia en cargos afecta exclusivamente a clientes con tenure=0, lo que bloquea el procesamiento numérico.

Se requiere conversión a float e imputación lógica para habilitar el análisis de rentabilidad si fuera requerido.

In [34]:
# Convierte espacios en blanco a NaN (nulo real)
df_telecom_norm['account.Charges.Total'] = df_telecom_norm['account.Charges.Total'].replace(' ', np.nan)

# Transforma a numérico.
df_telecom_norm['account.Charges.Total'] = pd.to_numeric(df_telecom_norm['account.Charges.Total'])

# Llena con el cargo mensual para no perder el registro.
df_telecom_norm['account.Charges.Total'] = df_telecom_norm['account.Charges.Total'].fillna(df_telecom_norm['account.Charges.Monthly'])

# Limpia el Churn.
df_telecom_norm = df_telecom_norm[df_telecom_norm['Churn'] != ' ']

# Verificación del pipeline de nuevo internamente.
print('**** Validación Post-Limpieza ****')
print(f'Nuevos espacios en blanco en Charges.Total: {df_telecom_norm[df_telecom_norm["account.Charges.Total"] == " "].shape[0]}')
print(f'Tipo de dato final de Charges.Total: {df_telecom_norm["account.Charges.Total"].dtype}')
print(f'Registros totales restantes: {len(df_telecom_norm)}')

**** Validación Post-Limpieza ****
Nuevos espacios en blanco en Charges.Total: 0
Tipo de dato final de Charges.Total: float64
Registros totales restantes: 7267


#### Nota 3: Se identificó que los registros con valores nulos en account.Charges.Total corresponden exclusivamente a altas nuevas (tenure=0). Dado que estos clientes ya tienen una renta contractual definida pero aún no completan su primer ciclo de facturación, se procedió a igualar el cargo total al cargo mensual (account.Charges.Monthly). Esta acción asegura la integridad de la base de datos y evita la subestimación de ingresos.

#### Nota 4. En Churn son solo 2 registros con espacio en blanco, lo más seguro es eliminarlos para no sesgar un modelo de predicción.

#📊 Carga y análisis

#📄Informe final